In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np
import pandas as pd
import json, time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
DS_LABEL = {'nsl_kdd_v2': 'NSL-KDD', 'unsw_nb15_v2': 'UNSW-NB15', 'cic_ids2017_v2': 'CIC-IDS2017'}
MODELS = [f'{a}_{v}' for v in ['5class_cw', '5class_smote'] for a in ['rf', 'xgb', 'dnn']]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_CLASSES = 5
ECE_N_BINS = 10          # as in 03d
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
TOP_K = 10

TABLES = Path(REPO) / 'results' / 'tables'
FIGS = Path(REPO) / 'results' / 'figures'
PREFIX = 'paper_inputs'

def find_proba_file(dataset, model_name, split):
    fname = f'{model_name}_{split}_proba.npy'
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / fname
        if p.exists():
            return p
    raise FileNotFoundError(f'No {fname} for {dataset}/{model_name}')
print('ready')


In [ ]:
def stratified_boot_indices(y, B, seed):
    # Resample within class so every resample keeps the test class proportions (03d convention).
    rng = np.random.RandomState(seed)
    groups = [np.where(y == c)[0] for c in np.unique(y)]
    out = np.empty((B, len(y)), dtype=np.int64)
    for b in range(B):
        pos = 0
        for g in groups:
            out[b, pos:pos + len(g)] = rng.choice(g, size=len(g), replace=True)
            pos += len(g)
    return out

def macro_f1_from_conf(conf):
    # conf: (5, 5) counts, rows true, cols predicted
    tp = np.diag(conf).astype(float)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
    return float(f1.mean()), f1

def f1_boot(y, pred, idx):
    code = y * N_CLASSES + pred
    conf0 = np.bincount(code, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
    point, per_class = macro_f1_from_conf(conf0)
    acc = float((y == pred).mean())
    B = idx.shape[0]
    vals, accs = np.empty(B), np.empty(B)
    for b in range(B):
        c = code[idx[b]]
        conf = np.bincount(c, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
        vals[b] = macro_f1_from_conf(conf)[0]
        accs[b] = float((y[idx[b]] == pred[idx[b]]).mean())
    return point, per_class, acc, vals, accs

def ece_binned(p, lab, bins, n_bins=ECE_N_BINS):
    # ECE on a (possibly resampled) binary problem from precomputed bin ids: sum_b (n_b/n)|mean p - mean y|.
    n = len(p)
    cnt = np.bincount(bins, minlength=n_bins).astype(float)
    sp = np.bincount(bins, weights=p, minlength=n_bins)
    sy = np.bincount(bins, weights=lab, minlength=n_bins)
    m = cnt > 0
    return float(np.sum(np.abs(sp[m] - sy[m])) / n)

def bin_ids(p, n_bins=ECE_N_BINS):
    # 03d edges: last bin closed on the right.
    return np.clip((p * n_bins).astype(int), 0, n_bins - 1)

def calib_metrics_boot(P_pre, P_post, y, idx):
    # Returns dict of arrays over resamples plus point estimates.
    res = {}
    Y = np.stack([(y == c).astype(float) for c in range(N_CLASSES)], axis=1)
    pred_pre, pred_post = P_pre.argmax(axis=1), P_post.argmax(axis=1)
    top_pre, top_post = P_pre.max(axis=1), P_post.max(axis=1)
    corr_pre, corr_post = (pred_pre == y).astype(float), (pred_post == y).astype(float)
    bins = {('macro', tag, c): bin_ids(P[:, c]) for tag, P in [('pre', P_pre), ('post', P_post)] for c in range(N_CLASSES)}
    bins[('top', 'pre')], bins[('top', 'post')] = bin_ids(top_pre), bin_ids(top_post)

    def all_metrics(sel):
        out = {}
        for tag, P, top, corr in [('pre', P_pre, top_pre, corr_pre), ('post', P_post, top_post, corr_post)]:
            Ps, Ys = P[sel], Y[sel]
            out[f'brier_macro_{tag}'] = float(np.mean(((Ps - Ys) ** 2).mean(axis=0)))
            out[f'ece_macro_{tag}'] = float(np.mean([ece_binned(Ps[:, c], Ys[:, c], bins[('macro', tag, c)][sel]) for c in range(N_CLASSES)]))
            out[f'ece_top_{tag}'] = ece_binned(top[sel], corr[sel], bins[('top', tag)][sel])
        out['pct_argmax_flipped'] = 100.0 * float((pred_pre[sel] != pred_post[sel]).mean())
        return out

    point = all_metrics(np.arange(len(y)))
    B = idx.shape[0]
    boot = {k: np.empty(B) for k in point}
    for b in range(B):
        m = all_metrics(idx[b])
        for k in point:
            boot[k][b] = m[k]
    return point, boot

def ci(v):
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
print('helpers ready')


In [ ]:
t0 = time.time()
f1_rows, cal_rows, rel_curves = [], [], {}
for ds in DATASETS:
    y = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    idx = stratified_boot_indices(y, N_BOOTSTRAP, BOOTSTRAP_SEED)
    for m in MODELS:
        P_pre = np.load(find_proba_file(ds, m, 'test')).astype(np.float64)
        P_post = np.load(Path(REPO) / 'calibrators' / ds / f'{m}_test_proba_calibrated.npy').astype(np.float64)
        pred = P_pre.argmax(axis=1)
        point, per_class, acc, vals, accs = f1_boot(y, pred, idx)
        row = {'dataset': ds, 'model': m, 'n_test': len(y), 'macro_f1': point,
               'macro_f1_lo': ci(vals)[0], 'macro_f1_hi': ci(vals)[1],
               'accuracy': acc, 'accuracy_lo': ci(accs)[0], 'accuracy_hi': ci(accs)[1]}
        row.update({f'f1_{CLASS_NAMES_5[c]}': float(per_class[c]) for c in range(N_CLASSES)})
        f1_rows.append(row)
        pt, bt = calib_metrics_boot(P_pre, P_post, y, idx)
        crow = {'dataset': ds, 'model': m, 'n_test': len(y)}
        for k in pt:
            lo, hi = ci(bt[k]); crow.update({k: pt[k], f'{k}_lo': lo, f'{k}_hi': hi})
        for k in ['brier_macro', 'ece_macro', 'ece_top']:
            d = bt[f'{k}_post'] - bt[f'{k}_pre']; lo, hi = ci(d)
            crow.update({f'{k}_delta': pt[f'{k}_post'] - pt[f'{k}_pre'], f'{k}_delta_lo': lo, f'{k}_delta_hi': hi,
                         f'{k}_verdict': 'improved' if hi < 0 else ('worsened' if lo > 0 else 'no difference')})
        cal_rows.append(crow)
        # reliability curve (top-label, calibrated), 10 bins
        top, corr = P_post.max(axis=1), (P_post.argmax(axis=1) == y).astype(float)
        b = bin_ids(top)
        cnt = np.bincount(b, minlength=ECE_N_BINS)
        conf = np.bincount(b, weights=top, minlength=ECE_N_BINS) / np.maximum(cnt, 1)
        accb = np.bincount(b, weights=corr, minlength=ECE_N_BINS) / np.maximum(cnt, 1)
        rel_curves[(ds, m)] = (conf, accb, cnt)
        print(f'{ds:15s} {m:18s} F1={point:.4f} [{row["macro_f1_lo"]:.4f},{row["macro_f1_hi"]:.4f}]  '
              f'ECE_top {pt["ece_top_pre"]:.4f}->{pt["ece_top_post"]:.4f} ({crow["ece_top_verdict"]})  '
              f'ECE_macro {pt["ece_macro_pre"]:.4f}->{pt["ece_macro_post"]:.4f} ({crow["ece_macro_verdict"]})  '
              f'Brier_macro {pt["brier_macro_pre"]:.4f}->{pt["brier_macro_post"]:.4f} ({crow["brier_macro_verdict"]})')
df_f1 = pd.DataFrame(f1_rows); df_f1.to_csv(TABLES / f'{PREFIX}_macro_f1.csv', index=False)
df_cal = pd.DataFrame(cal_rows); df_cal.to_csv(TABLES / f'{PREFIX}_calibration.csv', index=False)
print(f'\n{(time.time() - t0) / 60:.1f} min')


In [ ]:
plt.rcParams.update({'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.4), sharey=True)
styles = {'rf': ('#000000', 'o'), 'xgb': ('#4477AA', 's'), 'dnn': ('#CC3311', '^')}
for ax, ds in zip(axes, DATASETS):
    for m in MODELS:
        conf, accb, cnt = rel_curves[(ds, m)]
        keep = cnt >= 20
        col, mk = styles[m.split('_')[0]]
        ls = '-' if m.endswith('cw') else '--'
        e = float(df_cal[(df_cal.dataset == ds) & (df_cal.model == m)].ece_top_post.iloc[0])
        ax.plot(conf[keep], accb[keep], marker=mk, ms=3, color=col, ls=ls, lw=0.9,
                label=f'{m.split("_")[0].upper()} {"CW" if m.endswith("cw") else "SMOTE"} (ECE {e:.3f})')
    ax.plot([0, 1], [0, 1], ls=':', color='#999999', lw=0.8)
    ax.set_title(DS_LABEL[ds]); ax.set_xlabel('Mean calibrated top-label probability'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(frameon=False, fontsize=5.5, loc='upper left')
    ax.grid(lw=0.3, alpha=0.5)
axes[0].set_ylabel('Empirical accuracy in bin')
fig.tight_layout()
fig.savefig(FIGS / f'{PREFIX}_reliability.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGS / f'{PREFIX}_reliability.pdf', bbox_inches='tight')
print('saved reliability figure (bins with fewer than 20 samples omitted)')


In [ ]:
rows = []
for ds in DATASETS:
    sv_dir = Path(REPO) / 'shap_values' / ds
    names = json.load(open(f'{REPO}/data/processed/{ds}/feature_names.json'))
    for variant in ['5class_cw', '5class_smote']:
        srcs = {'xgb (logit)': sv_dir / f'xgb_{variant}_shap_shared.npy',
                'rf (prob)': sv_dir / f'rf_{variant}_shap_shared.npy',
                'dnn (softmax)': sv_dir / f'dnn_{variant}_shap_shared.npy',
                'dnn (logit)': sv_dir / f'dnn_{variant}_shap_shared_logit.npy'}
        for src, path in srcs.items():
            arr = np.load(path)
            imp = np.abs(arr).sum(axis=-1).mean(axis=0)          # mean over canonical samples of class-summed |SHAP|
            order = np.argsort(-imp)
            for r, fi in enumerate(order[:TOP_K]):
                rows.append({'dataset': ds, 'variant': variant, 'source': src, 'rank': r + 1,
                             'feature_index': int(fi), 'feature': names[fi], 'mean_abs_shap': float(imp[fi]),
                             'share_of_total': float(imp[fi] / imp.sum())})
df_top = pd.DataFrame(rows)
df_top.to_csv(TABLES / f'{PREFIX}_top10_features.csv', index=False)

# Side-by-side view for the NSL claim: XGB's top-10 with each model's rank of those same features
for variant in ['5class_cw', '5class_smote']:
    g = df_top[(df_top.dataset == 'nsl_kdd_v2') & (df_top.variant == variant)]
    xgb10 = g[g.source == 'xgb (logit)'].sort_values('rank')
    print(f'\nNSL-KDD {variant}: XGB top-10 and where the DNN ranks those features')
    for _, r in xgb10.iterrows():
        line = f'  {r["rank"]:2d} {r.feature:28s} share={r.share_of_total:.3f}'
        for src in ['dnn (softmax)', 'dnn (logit)', 'rf (prob)']:
            arr = np.load(Path(REPO) / 'shap_values' / 'nsl_kdd_v2' / {'dnn (softmax)': f'dnn_{variant}_shap_shared.npy',
                                                                            'dnn (logit)': f'dnn_{variant}_shap_shared_logit.npy',
                                                                            'rf (prob)': f'rf_{variant}_shap_shared.npy'}[src])
            imp = np.abs(arr).sum(axis=-1).mean(axis=0)
            rank = int(np.where(np.argsort(-imp) == r.feature_index)[0][0]) + 1
            line += f'   {src}: rank {rank:3d}'
        print(line)
print(f'\nsaved {PREFIX}_top10_features.csv ({len(df_top)} rows)')


In [ ]:
summary = {'timestamp': datetime.now().isoformat(), 'notebook': '14_paper_inputs.ipynb',
           'n_bootstrap': N_BOOTSTRAP, 'bootstrap': 'stratified by class over the full test partition',
           'ece_definition': f'{ECE_N_BINS} equal-width bins; ece_top = top-label confidence vs correctness; ece_macro = mean over classes of one-vs-rest ECE (03d convention)',
           'outputs': sorted(p.name for p in TABLES.glob(f'{PREFIX}_*')) + [f'{PREFIX}_reliability.png']}
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
pd.set_option('display.width', 250)
print(df_f1[['dataset', 'model', 'macro_f1', 'macro_f1_lo', 'macro_f1_hi', 'accuracy']].round(4).to_string(index=False))
print()
print(df_cal[['dataset', 'model', 'ece_top_pre', 'ece_top_post', 'ece_top_verdict', 'ece_macro_pre', 'ece_macro_post', 'ece_macro_verdict',
              'brier_macro_pre', 'brier_macro_post', 'brier_macro_verdict', 'pct_argmax_flipped']].round(4).to_string(index=False))


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"

import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '14_paper_inputs.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')

!git add notebooks/14_paper_inputs.ipynb
!git add results/tables/paper_inputs_*.csv results/tables/paper_inputs_summary.json
!git add results/figures/paper_inputs_reliability.png results/figures/paper_inputs_reliability.pdf
!git status --short | head -30
!git commit -m "Notebook 14: manuscript inputs. Macro-F1 and accuracy with stratified bootstrap CIs on full test sets; calibration pre/post (top-label and macro ECE, macro Brier) with paired CIs; reliability figure; top-10 feature lists per SHAP source"
!git push origin main
!git log --oneline -3
